# Probability basics

## Why this matters

A model never says "this is the answer". It says "this is how likely each answer is". A spam filter does not declare an email spam, it puts 92% on spam and 8% on not. Treating that number as a real probability, rather than a score to threshold, is what lets you reason about risk and calibrate decisions later.

Think of weather: "70% chance of rain" is a probability, and over many such days it should rain on about 70 of every 100. The same idea runs through every model prediction.

Two knobs anchor the notebook. A density (pdf) is a rate, not a probability, so its height can exceed 1; only the area under it between two points is a probability. And the standard deviation of a sample mean shrinks like $\sigma/\sqrt{n}$: four times the data halves the error, not quarters it. This notebook builds the vocabulary: random variables, the four common distributions, expectation and variance, and what happens when you average many samples.

Order: intuition, maths, idea, from scratch, library, what breaks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

rng = np.random.default_rng(42)
np.set_printoptions(precision=4, suppress=True)

## 1. Intuition

A random variable is a number that comes out of a process you cannot predict exactly. A coin flip coded as 0 or 1. The height of the next person through the door. The number of emails in the next hour. You cannot name the value in advance, but you can describe the pattern it follows over many repeats.

That pattern is the distribution: it says how likely each value is. Discrete variables, which take separate values like 0, 1, 2, have a probability mass function (pmf) that gives a probability to each one. Continuous variables, which take any real value, have a density (pdf) where only areas under the curve are probabilities, never the height at a single point. That distinction trips people up: a density can be larger than 1, because it is a rate, not a probability.

Four distributions cover most of this path, and each one answers a different shape of question:

- Bernoulli: one yes/no trial. Spam or not.
- Binomial: the number of yeses in $n$ independent trials.
- Normal: sums and averages of many small effects. Measurement noise.
- Poisson: counts of rare events in a fixed window.

## 2. Maths

Bernoulli($p$): $P(X=1)=p$, mean $p$, variance $p(1-p)$.

Binomial($n,p$): $P(X=k)=\binom{n}{k}p^k(1-p)^{n-k}$, mean $np$, variance $np(1-p)$.

Normal($\mu,\sigma^2$): $f(x)=\frac{1}{\sigma\sqrt{2\pi}}\exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$.

Poisson($\lambda$): $P(X=k)=\frac{\lambda^k e^{-\lambda}}{k!}$, mean and variance both $\lambda$.

Expectation: $E[X]=\sum_x x\,P(X=x)$. Variance: $\mathrm{Var}(X)=E[(X-E[X])^2]=E[X^2]-E[X]^2$. Scaling: $\mathrm{Var}(aX+b)=a^2\mathrm{Var}(X)$.

Conditional probability: $P(A\mid B)=P(A,B)/P(B)$. Independence: $P(A,B)=P(A)P(B)$, which is the same as $P(A\mid B)=P(A)$.

## 3. Idea

Two results explain why averaging data is so powerful, and why nearly every estimate in ML is an average of something.

**The law of large numbers:** the sample mean of $n$ draws converges to the true mean as $n$ grows. Average enough rolls of a fair die and you get 3.5. This is the guarantee that more data pulls your estimate towards the truth.

**The central limit theorem (CLT):** the sample mean is approximately normal with standard deviation $\sigma/\sqrt{n}$, whatever the shape of the original distribution, as long as its variance is finite. Two things fall out of this. The bell curve appears everywhere because averages are everywhere. And the error shrinks like $1/\sqrt{n}$, not $1/n$, so quadrupling the data only halves the error. Precision gets expensive fast.

## 4. From scratch: the pmfs and pdf

Write each formula directly, then compare with scipy in the next section.

In [ ]:
from math import comb, exp, factorial, pi, sqrt


def binom_pmf(k: int, n: int, p: float) -> float:
    return comb(n, k) * p**k * (1 - p) ** (n - k)


def normal_pdf(x: float, mu: float, sigma: float) -> float:
    return exp(-((x - mu) ** 2) / (2 * sigma**2)) / (sigma * sqrt(2 * pi))


def poisson_pmf(k: int, lam: float) -> float:
    return lam**k * exp(-lam) / factorial(k)


print("P(3 heads in 10 fair flips):", binom_pmf(3, 10, 0.5))
print("normal pdf at the mean (0, 1):", normal_pdf(0, 0, 1))
print("P(2 emails) with lambda=4:", poisson_pmf(2, 4))

Expectation and variance straight from the definition, for a fair six-sided die. Then check by simulation.

In [ ]:
values = np.arange(1, 7)
probs = np.full(6, 1 / 6)
mean = (values * probs).sum()
var = ((values - mean) ** 2 * probs).sum()
print("E[X] =", mean, " Var(X) =", var)

rolls = rng.integers(1, 7, size=100_000)
print("simulated:", rolls.mean(), rolls.var())
print("Var(3X + 2) simulated:", (3 * rolls + 2).var(), " 9 * Var(X):", 9 * var)

## 5. Library: scipy.stats

Same numbers, fewer lines, and sampling, cdf and quantiles come free.

In [ ]:
print(stats.binom.pmf(3, 10, 0.5), binom_pmf(3, 10, 0.5))
print(stats.norm.pdf(0, 0, 1), normal_pdf(0, 0, 1))
print(stats.poisson.pmf(2, 4), poisson_pmf(2, 4))

# Share of a normal within 1, 2 and 3 standard deviations
for k in (1, 2, 3):
    print(f"within {k} sd: {stats.norm.cdf(k) - stats.norm.cdf(-k):.4f}")

fig, axes = plt.subplots(1, 3, figsize=(11, 3))
ks = np.arange(0, 21)
axes[0].bar(ks, stats.binom.pmf(ks, 20, 0.3))
axes[0].set_title("Binomial(20, 0.3)")
xs = np.linspace(-4, 4, 200)
axes[1].plot(xs, stats.norm.pdf(xs))
axes[1].set_title("Normal(0, 1)")
axes[2].bar(ks, stats.poisson.pmf(ks, 4))
axes[2].set_title("Poisson(4)")
plt.tight_layout()
plt.show()

Poisson is the limit of a binomial with many trials and a small chance each. With $n=1000$ and $p=0.004$ the mean is 4, and the two pmfs sit almost on top of each other.

In [ ]:
ks = np.arange(0, 12)
gap = np.abs(stats.binom.pmf(ks, 1000, 0.004) - stats.poisson.pmf(ks, 4)).max()
print("largest pmf gap:", gap)

## 6. Law of large numbers and CLT by simulation

In [ ]:
from matplotlib.axes import Axes


def plot_running_mean(
    running: np.ndarray, true_mean: float, ax: Axes | None = None
) -> Axes:
    """Plot a running average settling towards its true mean (law of large numbers).

    Takes a pre-computed running-mean array and the true mean. Draws no random
    numbers of its own. Returns the Axes so the caller can style it further.
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 3))
    ns = np.arange(1, running.size + 1)
    ax.plot(ns, running, lw=1)
    ax.axhline(true_mean, color="grey", ls=":", label=f"true mean {true_mean}")
    ax.set_xscale("log")
    ax.set_xlabel("n (log scale)")
    ax.set_ylabel("running mean")
    ax.set_title("Running mean converges (LLN)")
    ax.legend()
    return ax


# Sample data: 5000 die rolls, integers 1..6, uniform, as a 1-D int array.
# The running cumulative mean must settle at 3.5, the mean of a fair die.
# Seed is fixed so the printed values and the plot reproduce exactly.
rolls = rng.integers(1, 7, size=5000)
running = np.cumsum(rolls) / np.arange(1, 5001)
for n in (10, 100, 1000, 5000):
    print(f"n={n:>5}: mean = {running[n - 1]:.3f}")

plot_running_mean(running, true_mean=3.5)
plt.tight_layout()
plt.show()

# CLT: sample means of a skewed variable (exponential, mean 1, sd 1).
# Each panel draws 10,000 means, each an average of n draws, as a float array.
# n = 1, 5, 50 is chosen to show the shape move from skewed (n=1, the raw
# exponential) to near-normal (n=50) as averaging washes out the original shape.
# Seed is fixed so the histograms and skew values reproduce exactly.
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, n in zip(axes, (1, 5, 50)):
    means = rng.exponential(1.0, size=(10_000, n)).mean(axis=1)
    ax.hist(means, bins=50, density=True, alpha=0.6)
    xs = np.linspace(means.min(), means.max(), 200)
    ax.plot(xs, stats.norm.pdf(xs, 1.0, 1.0 / np.sqrt(n)))
    ax.set_title(f"n={n}, skew={stats.skew(means):.2f}")
plt.tight_layout()
plt.show()

The skew of the raw exponential is 2, strongly lopsided. By $n=50$ it is close to 0.3 and the normal curve fits the histogram well, which is the CLT at work: averaging washes out the shape of the original. The standard deviation of the mean follows $1/\sqrt{n}$, and that single formula explains a lot of real numbers. It is why a poll of 1000 people carries roughly 3 percentage points of error, and why pushing that to 4000 people, four times the cost, only halves the error to about 1.5 points.

## 7. Joint and conditional probability, independence

A small table of 1000 synthetic customers: did they get a discount, did they buy.

In [ ]:
import pandas as pd

# Sample data: 1000 synthetic customers, two boolean columns.
# discount = did they get a discount (about 40% do); buy = did they purchase.
# Buy rate is set to 0.30 with a discount and 0.10 without, so the two are
# dependent by construction and the conditional and joint checks below differ.
# Seed is fixed so the printed probabilities and the chi-squared test reproduce.
n = 1000
discount = rng.random(n) < 0.4
buy = rng.random(n) < np.where(discount, 0.30, 0.10)
df = pd.DataFrame({"discount": discount, "buy": buy})

joint = pd.crosstab(df["discount"], df["buy"], normalize="all")
print(joint)

p_d = df["discount"].mean()
p_b = df["buy"].mean()
p_both = (df["discount"] & df["buy"]).mean()
print("P(buy)               :", p_b)
print("P(buy | discount)    :", p_both / p_d)
print("P(discount) * P(buy) :", p_d * p_b, " vs joint:", p_both)

If the two were independent the last two numbers would match. A chi-squared test turns the gap into a p-value.

In [ ]:
chi2, p_value, _, _ = stats.chi2_contingency(pd.crosstab(df["discount"], df["buy"]))
print(f"discount vs buy: chi2={chi2:.1f}, p={p_value:.2g}")

# Control data: a fair coin flip (about 50% heads), one boolean per customer,
# generated with no link to discount. It stands in for a genuinely independent
# variable, so the test below should fail to find a relationship.
# Seed is fixed so the control p-value reproduces.
coin = rng.random(n) < 0.5
chi2, p_value, _, _ = stats.chi2_contingency(pd.crosstab(df["discount"], coin))
print(f"discount vs coin: chi2={chi2:.2f}, p={p_value:.2f}")

## 8. What breaks

In [ ]:
# Sample data for two failure cases.
# Cauchy: 100,000 standard-Cauchy draws, a float array with heavy tails and no
# finite mean, so its running mean never settles. Chosen to break the LLN.
# iid vs copies: two 5000-by-100 float arrays of means. iid averages 100
# independent draws; copies averages 100 repeats of one draw, so it carries one
# observation's worth of information despite the same shape.
# Seed is fixed so the printed running means and standard errors reproduce.

# 1. Heavy tails: the Cauchy distribution has no mean, so the LLN does not apply
cauchy = rng.standard_cauchy(100_000)
running = np.cumsum(cauchy) / np.arange(1, cauchy.size + 1)
for n in (100, 10_000, 100_000):
    print(f"Cauchy running mean at n={n:>6}: {running[n - 1]:8.3f}")

# 2. Dependent samples: 100 copies of the same value look like n=100 but carry one observation
iid = rng.normal(0, 1, size=(5000, 100)).mean(axis=1)
copies = np.repeat(rng.normal(0, 1, size=(5000, 1)), 100, axis=1).mean(axis=1)
print("sd of mean, independent:", iid.std(), " expected 0.1")
print("sd of mean, duplicated :", copies.std(), " expected 1.0")

The Cauchy mean never settles, it just jumps whenever a huge value lands, because the distribution has no finite mean for the average to converge to. The duplicated rows give a standard error ten times too small: 100 copies of one draw carry exactly one observation's worth of information, not a hundred. Both mistakes have the same symptom in practice, a model that looks more certain than it has any right to be. They show up as heavy-tailed data fitted with squared error, and as leaked or repeated rows shared between the training and test sets.

Next: Bayes' theorem, which turns these distributions into a way of updating beliefs as evidence arrives.